### Purpose:
Read the RFM output from the `SQL analysis` and visualise segment distribution.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

BASE_DIR = Path.cwd().parent

# Read environment variables
db_path_env = os.getenv("DB_PATH")

if db_path_env is None:
    raise ValueError("DB_PATH not found in .env")

DB_PATH = BASE_DIR / db_path_env

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")

con = duckdb.connect(str(DB_PATH), read_only=True)

### Load RFM output

In [ ]:
df = con.execute("""
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        rfm_score,
        segment
    FROM marts.fact_rfm
    WHERE segment IS NOT NULL
""").df()

SEGMENT_ORDER = ['Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']

df['segment'] = df['segment'].astype(str)
df['segment'] = pd.Categorical(df['segment'], categories=SEGMENT_ORDER, ordered=True)

### Chart - Customer count per segment

In [ ]:
seg_counts = df['segment'].value_counts().reindex(SEGMENT_ORDER)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']

bars = ax.bar(seg_counts.index, seg_counts.to_numpy(), color=colors, width=0.6)

ax.bar_label(bars, labels=[f'{v:,.0f}' for v in seg_counts.to_numpy()], 
             padding=4, fontsize=11)

ax.set_title('Customer count by RFM segment', fontsize=14, pad=12)
ax.set_xlabel('Segment')
ax.set_ylabel('Customers')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "segment_distribution.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

### Chart - Segment profile (grouped bar: recency, frequency, monetary)

In [ ]:
profile = df.groupby('segment', observed=True).agg(
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).reindex(SEGMENT_ORDER)

profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

x = range(len(SEGMENT_ORDER))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar([i - width for i in x], profile_norm['avg_recency'], width, label='Avg recency (normalised)', color='steelblue')
ax.bar(x, profile_norm['avg_frequency'], width, label='Avg frequency (normalised)', color='orange')
ax.bar([i + width for i in x], profile_norm['avg_monetary'], width, label='Avg monetary (normalised)', color='green')

ax.set_xticks(x)

ax.set_xticklabels(SEGMENT_ORDER)

ax.set_title('RFM segment profile — normalised averages', fontsize=14, pad=12)

ax.set_ylabel('Normalised score (0-1)')

ax.legend()

plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "segment_profile.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

### Chart - Frequency vs Monetary scatter coloured by segment

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
palette = {'Champions': '#2ecc71', 'Loyal': '#27ae60', 'Potential': '#f39c12',
           'At Risk': '#e67e22', 'Lost': '#e74c3c'}

for seg in SEGMENT_ORDER:
    sub = df[df['segment'] == seg]
    ax.scatter(sub['frequency'], sub['monetary'], label=seg,
               color=palette[seg], alpha=0.5, s=20)

ax.set_title('Frequency vs monetary value - coloured by segment', fontsize=14, pad=12)
ax.set_xlabel('Order frequency (count)')
ax.set_ylabel('Total monetary value (BRL)')
ax.legend(title='Segment')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'R${x:,.0f}'))
plt.tight_layout()
output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "rfm_scatter.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

### Revenue concentration

In [ ]:
total_revenue = df['monetary'].sum()
for seg in SEGMENT_ORDER:
    sub = df[df['segment'] == seg]
    pct_customers = len(sub) / len(df) * 100
    pct_revenue = sub['monetary'].sum() / total_revenue * 100
    print(f"{seg:12} | {len(sub):6,} customers ({pct_customers:.1f}%) | R${sub['monetary'].sum():10,.0f} ({pct_revenue:.1f}% of revenue)")